In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)

from src.model import SessionGRU, next_item_loss

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)

X = torch.from_numpy(np.load(DATA_PROCESSED / "train_seqs.npy"))
N_ITEMS = int(X.max())  # 1,855,603 (IDs are shifted by +1)

model = SessionGRU(N_ITEMS, dim=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

n_params = sum(p.numel() for p in model.parameters())
print("device:", device)
print("sequences:", X.shape)
print("parameters:", f"{n_params:,}")

device: mps
sequences: torch.Size([9716693, 20])
parameters: 118,783,616


In [2]:
BATCH = 1024

def train_steps(n_steps, log_every=50):
    model.train()
    perm = torch.randperm(len(X))
    losses, start = [], time.time()
    for step in range(n_steps):
        idx = perm[(step * BATCH) % len(X):(step * BATCH) % len(X) + BATCH]
        seq = X[idx].to(device)

        opt.zero_grad()
        loss = next_item_loss(model, seq)
        loss.backward()
        opt.step()

        losses.append(loss.item())
        if (step + 1) % log_every == 0:
            elapsed = time.time() - start
            print(f"step {step + 1:5d} | loss {np.mean(losses[-log_every:]):.3f} | {elapsed:.0f}s")
    return losses

losses = train_steps(500)

step    50 | loss 9.031 | 7s
step   100 | loss 8.896 | 13s
step   150 | loss 8.765 | 19s
step   200 | loss 8.655 | 24s
step   250 | loss 8.580 | 30s
step   300 | loss 8.538 | 36s
step   350 | loss 8.484 | 41s
step   400 | loss 8.456 | 47s
step   450 | loss 8.425 | 53s
step   500 | loss 8.416 | 58s


In [3]:
torch.manual_seed(42)
model = SessionGRU(N_ITEMS, dim=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

losses = train_steps(500)

step    50 | loss 8.720 | 6s
step   100 | loss 8.481 | 12s
step   150 | loss 8.404 | 17s
step   200 | loss 8.331 | 23s
step   250 | loss 8.271 | 28s
step   300 | loss 8.211 | 34s
step   350 | loss 8.130 | 39s
step   400 | loss 8.057 | 45s
step   450 | loss 7.987 | 50s
step   500 | loss 7.933 | 56s


In [4]:
steps_per_epoch = len(X) // BATCH
print("steps per epoch:", steps_per_epoch)

losses = train_steps(steps_per_epoch, log_every=500)

torch.save(model.state_dict(), MODELS / "session_gru_epoch1.pt")
print("saved")

steps per epoch: 9488
step   500 | loss 7.545 | 56s
step  1000 | loss 6.976 | 111s
step  1500 | loss 6.539 | 166s
step  2000 | loss 6.221 | 220s
step  2500 | loss 5.954 | 275s
step  3000 | loss 5.767 | 329s
step  3500 | loss 5.605 | 383s
step  4000 | loss 5.470 | 438s
step  4500 | loss 5.362 | 492s
step  5000 | loss 5.269 | 546s
step  5500 | loss 5.188 | 600s
step  6000 | loss 5.119 | 654s
step  6500 | loss 5.058 | 709s
step  7000 | loss 5.004 | 763s
step  7500 | loss 4.956 | 817s
step  8000 | loss 4.909 | 871s
step  8500 | loss 4.882 | 925s
step  9000 | loss 4.839 | 979s
saved


In [5]:
import pandas as pd

# 1. Build test sequences in the same format as training
ti = pd.read_parquet(DATA_PROCESSED / "test_input", columns=["session", "aid", "ts"])
ti = ti.sort_values(["session", "ts"], kind="stable")

s, a = ti["session"].to_numpy(), ti["aid"].to_numpy()
repeat = np.concatenate([[False], (s[1:] == s[:-1]) & (a[1:] == a[:-1])])
ti = ti[~repeat]
ti = ti[ti.groupby("session").cumcount(ascending=False) < 20]

sessions, first_idx, lengths = np.unique(ti["session"].to_numpy(), return_index=True, return_counts=True)
values = ti["aid"].to_numpy().astype(np.int32) + 1
n = len(sessions)
T = np.zeros((n, 20), dtype=np.int32)
rows = np.repeat(np.arange(n), lengths)
pos = np.arange(len(values)) - np.repeat(first_idx, lengths)
T[rows, pos + (20 - np.repeat(lengths, lengths))] = values
print("test sequences:", T.shape)

# 2. Score every item for each session, keep the top 40
K, B = 40, 256
model.eval()
cands = np.zeros((n, K), dtype=np.int32)
start = time.time()
with torch.no_grad():
    item_vecs = model.item_vectors()
    for i in range(0, n, B):
        sv = model.encode(torch.from_numpy(T[i:i + B]).to(device))
        scores = sv @ item_vecs.T
        scores[:, 0] = -float("inf")  # never recommend padding
        cands[i:i + B] = scores.topk(K).indices.cpu().numpy() - 1  # back to original aid
print(f"candidates done in {time.time() - start:.0f}s")

pd.DataFrame({"session": sessions, "gru": list(cands)}).to_parquet(
    DATA_PROCESSED / "gru_candidates.parquet"
)
print(pd.read_parquet(DATA_PROCESSED / "gru_candidates.parquet").head(3))

test sequences: (1801251, 20)
candidates done in 7378s
    session                                                gru
0  11098528  [901572, 1679529, 443797, 418916, 1714514, 163...
1  11098529  [1105029, 238164, 347960, 1560778, 1694360, 63...
2  11098530  [264500, 93444, 1677181, 1058185, 583026, 4092...


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CUTOFF = 1661119200
TWO_WEEKS = 14 * 24 * 3600

# item activity in the 2 weeks before the test cutoff
ev = pq.read_table(DATA_PROCESSED / "split_train", columns=["aid", "ts"]).to_pandas()
recent = ev.loc[ev["ts"] >= CUTOFF - TWO_WEEKS, "aid"].to_numpy()
del ev
counts = np.bincount(recent, minlength=1_855_603)

# all right answers in the test week (next clicks, carts, orders)
lb = pd.read_parquet(DATA_PROCESSED / "test_labels")
answers = {
    "clicks": lb["click"].dropna().astype(int).to_numpy(),
    "carts": np.concatenate([a for a in lb["carts"].dropna()]),
    "orders": np.concatenate([a for a in lb["orders"].dropna()]),
}

rows = []
for n_min in [1, 2, 3, 5, 10, 20]:
    keep = counts >= n_min
    row = {"min interactions": n_min, "items kept": int(keep.sum())}
    for name, a in answers.items():
        row[f"{name} covered"] = keep[a].mean()
    rows.append(row)

pd.DataFrame(rows).round(4)

,min interactions,items kept,clicks covered,carts covered,orders covered
0,1,1786386,0.9818,0.9751,0.9902
1,2,1718656,0.9760,0.9690,0.9856
2,3,1621655,0.9692,0.9620,0.9797
3,5,1382655,0.9547,0.9472,0.9675
4,10,949046,0.9240,0.9141,0.9396
5,20,603054,0.8780,0.8620,0.8954


In [2]:
import sys, time
import torch

sys.path.insert(0, str(PROJECT_ROOT))
from src.model import SessionGRU

device = torch.device("mps")
N_ITEMS = 1_855_603
model = SessionGRU(N_ITEMS, dim=64).to(device)
model.load_state_dict(torch.load(PROJECT_ROOT / "models" / "session_gru_epoch1.pt", map_location=device))
model.eval()

ti = pd.read_parquet(DATA_PROCESSED / "test_input", columns=["session", "aid", "ts"])
ti = ti.sort_values(["session", "ts"], kind="stable")
s, a = ti["session"].to_numpy(), ti["aid"].to_numpy()
repeat = np.concatenate([[False], (s[1:] == s[:-1]) & (a[1:] == a[:-1])])
ti = ti[~repeat]
ti = ti[ti.groupby("session").cumcount(ascending=False) < 20]
sessions, first_idx, lengths = np.unique(ti["session"].to_numpy(), return_index=True, return_counts=True)
values = ti["aid"].to_numpy().astype(np.int32) + 1
n = len(sessions)
T = np.zeros((n, 20), dtype=np.int32)
rows_ = np.repeat(np.arange(n), lengths)
pos = np.arange(len(values)) - np.repeat(first_idx, lengths)
T[rows_, pos + (20 - np.repeat(lengths, lengths))] = values
print("test sequences:", T.shape)

test sequences: (1801251, 20)


In [3]:
B, K = 256, 40
n_batches = (n + B - 1) // B

with torch.no_grad():
    item_vecs = model.item_vectors()           # on GPU, float32
    item_vecs_half = item_vecs.half()          # on GPU, float16
    item_vecs_cpu = item_vecs.cpu()            # on CPU, float32
    model_cpu = SessionGRU(N_ITEMS, dim=64)
    model_cpu.load_state_dict(model.state_dict())
    model_cpu.eval()

    def a_gpu_topk(batch):
        sv = model.encode(batch.to(device))
        return (sv @ item_vecs.T).topk(K).indices.cpu()

    def b_cpu_topk(batch):
        sv = model.encode(batch.to(device))
        return (sv @ item_vecs.T).cpu().topk(K).indices

    def c_half_cpu_topk(batch):
        sv = model.encode(batch.to(device)).half()
        return (sv @ item_vecs_half.T).cpu().float().topk(K).indices

    def d_all_cpu(batch):
        sv = model_cpu.encode(batch)
        return (sv @ item_vecs_cpu.T).topk(K).indices

    for name, fn in [("A: GPU score + GPU top-k", a_gpu_topk),
                     ("B: GPU score + CPU top-k", b_cpu_topk),
                     ("C: GPU half + CPU top-k", c_half_cpu_topk),
                     ("D: all on CPU", d_all_cpu)]:
        fn(torch.from_numpy(T[:B]))  # warm-up
        torch.mps.synchronize()
        t0 = time.time()
        for i in range(0, 20 * B, B):
            fn(torch.from_numpy(T[i:i + B]))
        torch.mps.synchronize()
        per = (time.time() - t0) / 20
        print(f"{name}: {per:.3f}s per batch -> about {per * n_batches / 60:.0f} min total")

A: GPU score + GPU top-k: 1.243s per batch -> about 146 min total
B: GPU score + CPU top-k: 0.343s per batch -> about 40 min total
C: GPU half + CPU top-k: 0.286s per batch -> about 34 min total
D: all on CPU: 0.318s per batch -> about 37 min total


In [4]:
from src.model import next_item_loss
from src.retrieval import build_sequences

torch.manual_seed(42)

# 1. sequences from weeks 1-2 only
ev2 = pq.read_table(DATA_PROCESSED / "rank_train", columns=["session", "aid", "ts"]).to_pandas()
_, seqs2 = build_sequences(ev2)
del ev2
seqs2 = seqs2[seqs2[:, -2] > 0]  # need at least 2 items to learn a "next item"
X2 = torch.from_numpy(seqs2)
print("training sequences:", X2.shape)

# 2. fresh model, same settings as before
model2 = SessionGRU(N_ITEMS, dim=64).to(device)
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-2)
BATCH = 1024
steps = len(X2) // BATCH

model2.train()
perm = torch.randperm(len(X2))
losses, start = [], time.time()
for step in range(steps):
    seq = X2[perm[step * BATCH:(step + 1) * BATCH]].to(device)
    opt2.zero_grad()
    loss = next_item_loss(model2, seq)
    loss.backward()
    opt2.step()
    losses.append(loss.item())
    if (step + 1) % 500 == 0:
        print(f"step {step + 1}/{steps} | loss {np.mean(losses[-500:]):.3f} | {(time.time() - start) / 60:.1f} min")

torch.save(model2.state_dict(), PROJECT_ROOT / "models" / "session_gru_weeks12.pt")
print("saved")

training sequences: torch.Size([7099054, 20])
step 500/6932 | loss 8.215 | 1.1 min
step 1000/6932 | loss 7.495 | 2.0 min
step 1500/6932 | loss 6.904 | 2.9 min
step 2000/6932 | loss 6.457 | 3.8 min
step 2500/6932 | loss 6.125 | 4.7 min
step 3000/6932 | loss 5.874 | 5.6 min
step 3500/6932 | loss 5.675 | 6.5 min
step 4000/6932 | loss 5.522 | 7.4 min
step 4500/6932 | loss 5.385 | 8.3 min
step 5000/6932 | loss 5.275 | 9.2 min
step 5500/6932 | loss 5.179 | 10.2 min
step 6000/6932 | loss 5.105 | 11.2 min
step 6500/6932 | loss 5.044 | 12.1 min
saved
